# Quickstart: one callable, every input shape

screamer's defining feature: **the same configured function works on a scalar,
a NumPy array, a 2-D batch, a Python list, or a live iterator — and the numbers
are identical across all of them.** That's what lets you backtest on arrays and
run live on a stream with the same code and no train/serve skew.

In [ ]:
import numpy as np
from screamer import RollingMean

rng = np.random.default_rng(0)
x = rng.standard_normal(20)

## Scalars, arrays, and 2-D batches
`axis=0` is always time; extra axes are independent parallel series.

In [ ]:
ma = RollingMean(5)
print("scalar :", ma(1.5))                 # -> a float
print("array  :", RollingMean(5)(x)[:6])   # -> a (20,) array
batch = np.random.default_rng(1).standard_normal((20, 3))
print("2-D    :", RollingMean(5)(batch).shape)   # (20, 3): 3 independent columns

## Lists and live iterators
A list is processed eagerly; an iterator is processed lazily, one value at a time
(this is the live-event path).

In [ ]:
print("list   :", RollingMean(5)([1.0, 2.0, 3.0, 4.0, 5.0]))

def live_source(vals):
    for v in vals:            # imagine this yielding from a socket / clock
        yield v

live_ma = RollingMean(5)      # keep a reference; LazyIterator borrows from it
streamed = list(live_ma(live_source(x)))

## Batch == streaming (the key guarantee)

In [ ]:
batch_result = RollingMean(5)(x)
np.testing.assert_array_equal(batch_result, np.array(streamed))
print("batch == streamed:", np.allclose(batch_result, streamed, equal_nan=True))

**Takeaways**
- One configured functor handles scalar / array / 2-D / list / iterator.
- `axis=0` is time; higher axes are independent series (no `axis=` argument).
- The array (backtest) and iterator (live) paths give byte-identical results.